In [ ]:
# input text that the model will be trained on
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
# all unique characters
chars = sorted((list(set(text))))
vocab_size = len(chars)

In [ ]:
# create mapping form characters to integers
# Other mappings: google uses sentencepiece, openai uses tiktoken
# Balance between length of integers with length of vocab
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]   ## take string, output list of integers
decode = lambda l: ''.join([itos[i] for i in l])  ## take list of integers, output a string

In [ ]:
# now can encode dataset and into a tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)

In [ ]:
# split into train and validation sets
# helps understand whether its overfitting
n = int(0.9*len(data))
train_data = data[:n]  ## first 90 percent
val_data = data[n:]

In [ ]:
# size of input
block_size = 8
train_data[:block_size+1]

In [ ]:
# reducing block size means efficienct and for model to be familiar with any block size input up to "block_size"
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} that target: {target}")

In [ ]:
# mini batches of blocks since GPUs are very good as parallel processing
torch.manual_seed(1337)
batch_size = 4  ## how many independent sequences will be processed in parallel
block_size = 8  ## what is the maximum context length for predictions

def get_batch(split):
    # generate a small batch
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))  ## random integers of size batch_size
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size):  ## batch dimension
    for t in range(block_size):  ## time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")